# 07 marker de

**Purpose:** Calculate cluster marker genes before annotation.

**Original analysis notebook:** `07_marker_DE_pre_annotation_HVG8000_r0_6_fullgene_v3_fixed_marker_panel.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.


# 23 — Marker DE before annotation: full-gene, HVG8000/scVI r0.6

**Goal:** extract robust marker evidence for each cluster before manual/marker-based annotation.

This notebook intentionally performs differential marker ranking on the **full-gene object** with 15,176 genes, while using cluster labels transferred from the HVG8000/scVI integration.

Main clustering key:

```text
leiden_scvi_main
```

which is an alias for:

```text
leiden_scvi_HVG8000_r0_6
```

Outputs include:

- 100 upregulated genes per cluster
- 100 downregulated genes per cluster
- p-value and adjusted p-value
- Wilcoxon score and log fold-change
- expression fraction in cluster and rest, when available
- marker panel presence report
- QC reports and marker plots

Important design choices:

1. **Do not use integrated/scVI latent values for DE.** Use expression from full-gene data.
2. **Keep raw counts preserved** in `adata.layers['counts']` when available.
3. **Normalize/log1p a working copy** for marker ranking.
4. **Do not force cluster number.** This notebook uses the selected clustering from the previous QC step.


In [ ]:

# =========================
# Imports and global settings
# =========================

from pathlib import Path
import os
import gc
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp

import scanpy as sc
import anndata as ad

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
INTEGRATED_DIR = PROCESSED_DIR / "integrated"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures"
LOGS_DIR = PROJECT_DIR / "logs"

for d in [INTEGRATED_DIR, METADATA_DIR, FIGURES_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MARKER_FIG_DIR = FIGURES_DIR / "marker_DE_HVG8000_r0_6_pre_annotation"
MARKER_FIG_DIR.mkdir(parents=True, exist_ok=True)

sc.settings.figdir = str(MARKER_FIG_DIR)

print("Project:", PROJECT_DIR)
print("Figure dir:", MARKER_FIG_DIR)


In [ ]:

# =========================
# Parameters
# =========================

N_TOP_DIRECTION = 100
N_TOP_PLOT_PER_CLUSTER = 3

MAIN_CLUSTER_KEY = "leiden_scvi_main"
MAIN_CLUSTER_ORIGINAL_KEY = "leiden_scvi_HVG8000_r0_6"
REFINEMENT_CLUSTER_KEY = "leiden_scvi_HVG8000_r0_7"

DATASET_COL = "core_dataset"
CONDITION_COL = "condition"
PATIENT_COL = "patient_id"
SAMPLE_COL = "sample_id"

# Candidate full-gene objects. The first existing path will be used.
CANDIDATE_FULLGENE_H5ADS = [
    INTEGRATED_DIR / "GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad",
    INTEGRATED_DIR / "GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters_r0_6.h5ad",
]

# Main output h5ad for the next step.
MARKER_READY_H5AD = INTEGRATED_DIR / "GBM_core_4datasets_fullgene_HVG8000_r0_6_marker_DE_ready.h5ad"

# Metadata outputs.
DE_READINESS_TSV = METADATA_DIR / "marker_DE_HVG8000_r0_6_readiness_report.tsv"
CLUSTER_QC_TSV = METADATA_DIR / "marker_DE_HVG8000_r0_6_cluster_QC_summary.tsv"
CLUSTER_COMPOSITION_TSV = METADATA_DIR / "marker_DE_HVG8000_r0_6_cluster_composition_summary.tsv"
MARKER_PANEL_PRESENCE_TSV = METADATA_DIR / "marker_DE_HVG8000_r0_6_marker_panel_presence.tsv"

DE_ALL_TSV = METADATA_DIR / "marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_all_genes.tsv"
DE_TOP100_UP_DOWN_TSV = METADATA_DIR / "marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top100_up_down.tsv"
DE_TOP20_UP_TSV = METADATA_DIR / "marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top20_up.tsv"
DE_FILTERED_TSV = METADATA_DIR / "marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_filtered_padj005_logfc025.tsv"
DE_OUTPUT_MANIFEST_TSV = METADATA_DIR / "marker_DE_HVG8000_r0_6_output_manifest.tsv"

# Scanpy rank_genes_groups key
RANK_KEY = "rank_genes_groups_leiden_scvi_main_HVG8000_r0_6_wilcoxon"

print("Main cluster key:", MAIN_CLUSTER_KEY)
print("Original cluster key:", MAIN_CLUSTER_ORIGINAL_KEY)
print("Output marker TSV:", DE_TOP100_UP_DOWN_TSV)


In [ ]:

# =========================
# Helper functions
# =========================

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="	", index=False)
    print(f"[SAVED] {path} shape={df.shape}")


def write_h5ad_replace(adata_obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    adata_obj.write_h5ad(path)
    print(f"[SAVED] {path}")


def choose_first_existing(paths, label):
    print(f"Candidate {label} files:")
    existing = []
    for p in paths:
        p = Path(p)
        print(f"- {p} | exists: {p.exists()}")
        if p.exists():
            existing.append(p)
    if len(existing) == 0:
        raise FileNotFoundError(f"No {label} file found. Checked: " + ", ".join(map(str, paths)))
    print(f"\n[USING {label}] {existing[0]}")
    return existing[0]


def sparse_or_dense_sample_values(X, n_rows=1000, n_cols=1000):
    xs = X[:min(n_rows, X.shape[0]), :min(n_cols, X.shape[1])]
    if sp.issparse(xs):
        vals = xs.data
    else:
        vals = np.asarray(xs).ravel()
    vals = vals[np.isfinite(vals)]
    return vals


def looks_integer_like(X, n_rows=1000, n_cols=1000):
    vals = sparse_or_dense_sample_values(X, n_rows=n_rows, n_cols=n_cols)
    if len(vals) == 0:
        return True
    return bool(np.allclose(vals, np.round(vals)))


def matrix_min_max(X, n_rows=1000, n_cols=1000):
    vals = sparse_or_dense_sample_values(X, n_rows=n_rows, n_cols=n_cols)
    if len(vals) == 0:
        return np.nan, np.nan
    return float(vals.min()), float(vals.max())


def clean_old_rank_keys(adata_obj):
    for k in list(adata_obj.uns.keys()):
        if str(k).startswith("rank_genes_groups"):
            del adata_obj.uns[k]
            print(f"[REMOVED old uns key] {k}")


In [ ]:

# =========================
# Load full-gene object with transferred scVI clusters
# =========================

FULLGENE_INPUT_H5AD = choose_first_existing(CANDIDATE_FULLGENE_H5ADS, "full-gene input")

adata = sc.read_h5ad(FULLGENE_INPUT_H5AD)
print(adata)
print("layers:", list(adata.layers.keys()))
print("obsm:", list(adata.obsm.keys()))
print("obs columns preview:", list(adata.obs.columns)[:80])
print("var columns preview:", list(adata.var.columns)[:40])


In [ ]:

# =========================
# Validate required columns and create aliases if needed
# =========================

if MAIN_CLUSTER_KEY not in adata.obs.columns:
    if MAIN_CLUSTER_ORIGINAL_KEY in adata.obs.columns:
        adata.obs[MAIN_CLUSTER_KEY] = adata.obs[MAIN_CLUSTER_ORIGINAL_KEY].astype(str).values
        print(f"[CREATED alias] {MAIN_CLUSTER_KEY} from {MAIN_CLUSTER_ORIGINAL_KEY}")
    else:
        raise KeyError(
            f"Neither {MAIN_CLUSTER_KEY} nor {MAIN_CLUSTER_ORIGINAL_KEY} were found in adata.obs. "
            f"Available cluster-like columns: {[c for c in adata.obs.columns if 'leiden' in c.lower() or 'cluster' in c.lower()]}"
        )

adata.obs[MAIN_CLUSTER_KEY] = adata.obs[MAIN_CLUSTER_KEY].astype(str).astype("category")

required_obs = [MAIN_CLUSTER_KEY, DATASET_COL, CONDITION_COL, PATIENT_COL, SAMPLE_COL]
missing_required = [c for c in required_obs if c not in adata.obs.columns]
if len(missing_required) > 0:
    raise KeyError(f"Missing required obs columns: {missing_required}")

for c in required_obs:
    adata.obs[c] = adata.obs[c].astype(str).fillna("unknown").astype("category")

if "X_umap" not in adata.obsm.keys():
    print("[WARNING] X_umap not found. UMAP plots will be skipped.")
if "X_scVI" not in adata.obsm.keys():
    print("[WARNING] X_scVI not found. This is okay for marker DE, but check transfer step.")

cluster_counts = adata.obs[MAIN_CLUSTER_KEY].value_counts().sort_index()
print("Number of clusters:", cluster_counts.shape[0])
display(cluster_counts.to_frame("n_cells"))


In [ ]:

# =========================
# Readiness report
# =========================

x_min, x_max = matrix_min_max(adata.X)
counts_layer_present = "counts" in adata.layers
counts_integer_like = looks_integer_like(adata.layers["counts"]) if counts_layer_present else False
x_integer_like = looks_integer_like(adata.X)

readiness = pd.DataFrame([
    {"metric": "input_h5ad", "value": str(FULLGENE_INPUT_H5AD)},
    {"metric": "n_cells", "value": int(adata.n_obs)},
    {"metric": "n_genes", "value": int(adata.n_vars)},
    {"metric": "main_cluster_key", "value": MAIN_CLUSTER_KEY},
    {"metric": "n_clusters", "value": int(adata.obs[MAIN_CLUSTER_KEY].nunique())},
    {"metric": "counts_layer_present", "value": bool(counts_layer_present)},
    {"metric": "counts_layer_integer_like", "value": bool(counts_integer_like)},
    {"metric": "X_integer_like", "value": bool(x_integer_like)},
    {"metric": "X_sample_min", "value": x_min},
    {"metric": "X_sample_max", "value": x_max},
    {"metric": "X_umap_present", "value": "X_umap" in adata.obsm.keys()},
    {"metric": "X_scVI_present", "value": "X_scVI" in adata.obsm.keys()},
])
write_tsv_replace(readiness, DE_READINESS_TSV)
display(readiness)

if not counts_layer_present and not x_integer_like:
    raise ValueError(
        "No counts layer was found and adata.X does not look integer-like. "
        "For marker DE, this notebook expects raw counts either in adata.layers['counts'] or in adata.X."
    )


In [ ]:

# =========================
# Cluster size and composition QC
# =========================

cluster_size = (
    adata.obs[MAIN_CLUSTER_KEY]
    .value_counts()
    .rename_axis("cluster")
    .reset_index(name="n_cells")
)
cluster_size["cluster"] = cluster_size["cluster"].astype(str)
cluster_size = cluster_size.sort_values("cluster", key=lambda s: s.map(lambda x: int(x) if x.isdigit() else x))

# Dominant dataset / condition / patient / sample per cluster
def dominant_summary(obs, cluster_key, col):
    tab = pd.crosstab(obs[cluster_key].astype(str), obs[col].astype(str))
    frac = tab.div(tab.sum(axis=1), axis=0)
    out = pd.DataFrame({
        "cluster": tab.index.astype(str),
        f"top_{col}": frac.idxmax(axis=1).values,
        f"top_{col}_fraction": frac.max(axis=1).values,
        f"n_{col}": (tab > 0).sum(axis=1).values,
    })
    return out

qc = cluster_size.copy()
for col in [DATASET_COL, CONDITION_COL, PATIENT_COL, SAMPLE_COL]:
    qc = qc.merge(dominant_summary(adata.obs, MAIN_CLUSTER_KEY, col), on="cluster", how="left")

qc["fraction_total_cells"] = qc["n_cells"] / adata.n_obs
qc["small_cluster_lt_500"] = qc["n_cells"] < 500
qc["small_cluster_lt_100"] = qc["n_cells"] < 100
qc["dataset_dominant_gt_0_80"] = qc[f"top_{DATASET_COL}_fraction"] > 0.80
qc["condition_dominant_gt_0_80"] = qc[f"top_{CONDITION_COL}_fraction"] > 0.80
qc["patient_dominant_gt_0_80"] = qc[f"top_{PATIENT_COL}_fraction"] > 0.80
qc["sample_dominant_gt_0_80"] = qc[f"top_{SAMPLE_COL}_fraction"] > 0.80

write_tsv_replace(qc, CLUSTER_QC_TSV)
display(qc)

# Long composition table
composition_rows = []
for col in [DATASET_COL, CONDITION_COL, PATIENT_COL, SAMPLE_COL]:
    tab = pd.crosstab(adata.obs[MAIN_CLUSTER_KEY].astype(str), adata.obs[col].astype(str))
    frac = tab.div(tab.sum(axis=1), axis=0)
    for cluster in tab.index.astype(str):
        for value in tab.columns.astype(str):
            composition_rows.append({
                "cluster": cluster,
                "metadata_column": col,
                "metadata_value": value,
                "n_cells": int(tab.loc[cluster, value]),
                "fraction_within_cluster": float(frac.loc[cluster, value]),
            })
composition = pd.DataFrame(composition_rows)
write_tsv_replace(composition, CLUSTER_COMPOSITION_TSV)


In [ ]:

# =========================
# Prepare log-normalized expression for marker DE
# =========================

# We use full-gene expression, not scVI latent/integrated values.
# Scanpy rank_genes_groups expects log-normalized expression when use_raw=False.

adata_de = adata.copy()
clean_old_rank_keys(adata_de)
adata_de.raw = None

if "counts" in adata_de.layers:
    print("[USING] adata.layers['counts'] as raw counts source for normalization/log1p")
    adata_de.X = adata_de.layers["counts"].copy()
else:
    print("[USING] adata.X as raw counts source for normalization/log1p")

# Make sure X is float for normalization/log1p but keep sparsity.
if sp.issparse(adata_de.X):
    adata_de.X = adata_de.X.astype(np.float32)
else:
    adata_de.X = np.asarray(adata_de.X, dtype=np.float32)

# Basic sanity before normalization
vals = sparse_or_dense_sample_values(adata_de.X)
print("Raw-count source sample min/max:", vals.min() if len(vals) else None, vals.max() if len(vals) else None)
print("Raw-count source integer-like:", bool(np.allclose(vals, np.round(vals))) if len(vals) else True)

sc.pp.normalize_total(adata_de, target_sum=1e4)
sc.pp.log1p(adata_de)

adata_de.uns["marker_DE_preprocessing_HVG8000_r0_6"] = {
    "cluster_key": MAIN_CLUSTER_KEY,
    "counts_source": "adata.layers['counts']" if "counts" in adata.layers else "adata.X",
    "normalization": "scanpy.pp.normalize_total(target_sum=1e4)",
    "transformation": "scanpy.pp.log1p",
    "rank_method": "scanpy.tl.rank_genes_groups(method='wilcoxon', tie_correct=True, pts=True)",
    "use_raw": False,
    "note": "DE is performed on full-gene log-normalized expression, not on scVI latent values.",
}

print(adata_de)


In [ ]:

# =========================
# Save marker-DE-ready h5ad for reproducibility
# =========================

# This h5ad may be large. It is saved because each pipeline stage should be reproducible.
# The original full-gene object remains untouched.

write_h5ad_replace(adata_de, MARKER_READY_H5AD)


In [ ]:

# =========================
# Run Wilcoxon one-vs-rest marker ranking
# =========================

print("Running Scanpy rank_genes_groups")
print("groupby:", MAIN_CLUSTER_KEY)
print("method: wilcoxon")
print("n_genes:", adata_de.n_vars)

sc.tl.rank_genes_groups(
    adata_de,
    groupby=MAIN_CLUSTER_KEY,
    method="wilcoxon",
    corr_method="benjamini-hochberg",
    n_genes=adata_de.n_vars,
    use_raw=False,
    pts=True,
    tie_correct=True,
    rankby_abs=False,
    key_added=RANK_KEY,
)

print("[DONE] rank_genes_groups")
print("uns key:", RANK_KEY)


In [ ]:

# =========================
# Convert rank_genes_groups result to long table
# =========================

def _safe_df_from_uns_field(uns, field):
    if field not in uns:
        return None
    val = uns[field]
    try:
        return pd.DataFrame(val)
    except Exception:
        return None


def _get_pts_value(pts_obj, gene, group):
    if pts_obj is None:
        return np.nan
    try:
        if isinstance(pts_obj, pd.DataFrame):
            if gene in pts_obj.index and group in pts_obj.columns:
                return float(pts_obj.loc[gene, group])
        else:
            pts_df = pd.DataFrame(pts_obj)
            if gene in pts_df.index and group in pts_df.columns:
                return float(pts_df.loc[gene, group])
    except Exception:
        return np.nan
    return np.nan

rg = adata_de.uns[RANK_KEY]

names_df = pd.DataFrame(rg["names"])
scores_df = pd.DataFrame(rg["scores"])
logfc_df = pd.DataFrame(rg["logfoldchanges"])
pvals_df = pd.DataFrame(rg["pvals"])
padj_df = pd.DataFrame(rg["pvals_adj"])

pts = rg.get("pts", None)
pts_rest = rg.get("pts_rest", None)

rows = []
for group in names_df.columns.astype(str):
    for i in range(names_df.shape[0]):
        gene = str(names_df.loc[i, group])
        score = float(scores_df.loc[i, group]) if pd.notna(scores_df.loc[i, group]) else np.nan
        logfc = float(logfc_df.loc[i, group]) if pd.notna(logfc_df.loc[i, group]) else np.nan
        pval = float(pvals_df.loc[i, group]) if pd.notna(pvals_df.loc[i, group]) else np.nan
        padj = float(padj_df.loc[i, group]) if pd.notna(padj_df.loc[i, group]) else np.nan
        pct_in = _get_pts_value(pts, gene, group)
        pct_rest = _get_pts_value(pts_rest, gene, group)
        rows.append({
            "cluster": group,
            "rank_scanpy": i + 1,
            "gene": gene,
            "score": score,
            "logfoldchange": logfc,
            "pval": pval,
            "padj": padj,
            "pct_in_cluster": pct_in,
            "pct_rest": pct_rest,
            "pct_diff_in_minus_rest": pct_in - pct_rest if pd.notna(pct_in) and pd.notna(pct_rest) else np.nan,
        })

all_de = pd.DataFrame(rows)
all_de["direction"] = np.where(all_de["logfoldchange"] >= 0, "up", "down")
all_de["cluster"] = all_de["cluster"].astype(str)

write_tsv_replace(all_de, DE_ALL_TSV)
display(all_de.head())
print(all_de.shape)


In [ ]:

# =========================
# Select top 100 up and top 100 down per cluster
# =========================

top_tables = []
top20_up_tables = []
filtered_tables = []

for cluster, sub in all_de.groupby("cluster", sort=False):
    sub = sub.copy()

    # Upregulated: higher in the cluster than rest.
    up = sub[(sub["logfoldchange"] > 0) & (sub["score"] > 0)].copy()
    up = up.sort_values(["padj", "pval", "score", "logfoldchange"], ascending=[True, True, False, False])
    up = up.head(N_TOP_DIRECTION)
    up["direction"] = "up"
    up["direction_rank"] = np.arange(1, up.shape[0] + 1)
    top_tables.append(up)

    top20_up = up.head(20).copy()
    top20_up_tables.append(top20_up)

    # Downregulated: lower in the cluster than rest.
    down = sub[(sub["logfoldchange"] < 0) & (sub["score"] < 0)].copy()
    down = down.sort_values(["padj", "pval", "score", "logfoldchange"], ascending=[True, True, True, True])
    down = down.head(N_TOP_DIRECTION)
    down["direction"] = "down"
    down["direction_rank"] = np.arange(1, down.shape[0] + 1)
    top_tables.append(down)

    filt = sub[(sub["padj"] < 0.05) & (sub["logfoldchange"].abs() >= 0.25)].copy()
    filt = filt.sort_values(["padj", "pval", "score"], ascending=[True, True, False])
    filtered_tables.append(filt)

top100_up_down = pd.concat(top_tables, axis=0, ignore_index=True)
top20_up = pd.concat(top20_up_tables, axis=0, ignore_index=True)
filtered_de = pd.concat(filtered_tables, axis=0, ignore_index=True)

# Sort clusters naturally when possible.
def _cluster_sort_key(x):
    x = str(x)
    return int(x) if x.isdigit() else x

top100_up_down = top100_up_down.sort_values(
    ["cluster", "direction", "direction_rank"],
    key=lambda s: s.map(_cluster_sort_key) if s.name == "cluster" else s,
)

top20_up = top20_up.sort_values(
    ["cluster", "direction_rank"],
    key=lambda s: s.map(_cluster_sort_key) if s.name == "cluster" else s,
)

write_tsv_replace(top100_up_down, DE_TOP100_UP_DOWN_TSV)
write_tsv_replace(top20_up, DE_TOP20_UP_TSV)
write_tsv_replace(filtered_de, DE_FILTERED_TSV)

display(top100_up_down.head(30))
print("Top up/down rows:", top100_up_down.shape)
print("Filtered rows:", filtered_de.shape)


In [ ]:

# =========================
# Marker panel gene presence
# =========================

marker_panels = {
    "Tumor_AC_like": ["GFAP", "AQP4", "ALDH1L1", "SLC1A3", "CLU", "APOE"],
    "Tumor_OPC_like": ["PDGFRA", "OLIG1", "OLIG2", "SOX10", "CSPG4", "BCAN"],
    "Tumor_NPC_like": ["SOX2", "NES", "VIM", "FABP7", "DLL3", "ASCL1"],
    "Tumor_MES_like": ["CD44", "CHI3L1", "SERPINE1", "VIM", "TNC", "EMP3"],
    "Cycling": ["MKI67", "TOP2A", "UBE2C", "HMGB2", "STMN1", "CENPF"],
    "Hypoxia_stress": ["VEGFA", "CA9", "DDIT4", "HILPDA", "ENO1", "HMOX1"],
    "Myeloid": ["PTPRC", "TYROBP", "LST1", "C1QA", "C1QB", "C1QC", "CSF1R", "AIF1"],
    "Inflammatory_myeloid": ["IL1B", "CCL3", "CCL4", "CXCL8", "NFKBIA"],
    "T_NK": ["CD3D", "CD3E", "TRAC", "IL7R", "NKG7", "GNLY", "GZMB"],
    "B_Plasma": ["MS4A1", "CD79A", "MZB1", "JCHAIN", "SDC1"],
    "Oligodendrocyte": ["MBP", "PLP1", "MOG", "MAG", "MOBP"],
    "Endothelial": ["PECAM1", "VWF", "KDR", "FLT1", "CLDN5"],
    "Mural_Pericyte": ["PDGFRB", "RGS5", "ACTA2", "TAGLN", "MCAM"],
    "Fibroblast_ECM": ["COL1A1", "COL1A2", "DCN", "LUM", "COL3A1"],
}

var_set = set(map(str, adata_de.var_names))
presence_rows = []
for panel, genes in marker_panels.items():
    for gene in genes:
        presence_rows.append({
            "panel": panel,
            "gene": gene,
            "present_in_fullgene_object": gene in var_set,
        })
marker_presence = pd.DataFrame(presence_rows)
write_tsv_replace(marker_presence, MARKER_PANEL_PRESENCE_TSV)
display(marker_presence.groupby("panel")["present_in_fullgene_object"].agg(["sum", "count"]))


In [ ]:

# =========================
# Plots: UMAP and cluster sizes
# =========================

if "X_umap" in adata_de.obsm.keys():
    for color in [MAIN_CLUSTER_KEY, DATASET_COL, CONDITION_COL]:
        if color in adata_de.obs.columns:
            sc.pl.umap(
                adata_de,
                color=color,
                legend_loc="right margin",
                size=2,
                frameon=True,
                show=False,
            )
            fig = plt.gcf()
            out = MARKER_FIG_DIR / f"marker_DE_UMAP_{color}.png"
            fig.savefig(out, dpi=200, bbox_inches="tight")
            plt.close(fig)
            print("[SAVED]", out)
else:
    print("[SKIP] UMAP plots because X_umap is missing.")

# Cluster size barplot
fig, ax = plt.subplots(figsize=(10, 4))
plot_df = qc.sort_values("cluster", key=lambda s: s.map(lambda x: int(x) if str(x).isdigit() else x))
ax.bar(plot_df["cluster"].astype(str), plot_df["n_cells"])
ax.set_xlabel("Cluster")
ax.set_ylabel("Number of cells")
ax.set_title(f"Cluster sizes — {MAIN_CLUSTER_KEY}")
plt.xticks(rotation=45)
out = MARKER_FIG_DIR / "marker_DE_cluster_sizes.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.close(fig)
print("[SAVED]", out)


In [ ]:

# =========================
# Plots: top marker dotplot and known marker-panel dotplot
# =========================

# Top de novo markers: top N_TOP_PLOT_PER_CLUSTER upregulated genes per cluster.
plot_genes = []
for cluster, sub in top20_up.groupby("cluster", sort=False):
    genes = sub.sort_values("direction_rank")["gene"].head(N_TOP_PLOT_PER_CLUSTER).tolist()
    plot_genes.extend(genes)
plot_genes = list(dict.fromkeys([g for g in plot_genes if g in adata_de.var_names]))

if len(plot_genes) > 0:
    print("Number of top marker genes for dotplot:", len(plot_genes))
    sc.pl.dotplot(
        adata_de,
        var_names=plot_genes,
        groupby=MAIN_CLUSTER_KEY,
        standard_scale="var",
        dendrogram=False,
        show=False,
    )
    fig = plt.gcf()
    out = MARKER_FIG_DIR / "marker_DE_top3_up_per_cluster_dotplot.png"
    fig.savefig(out, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print("[SAVED]", out)
else:
    print("[SKIP] No top marker genes found for dotplot.")

# Known marker panel dotplot.
# IMPORTANT: Scanpy dotplot with dict var_names fails if even one gene is absent.
# Therefore, filter each marker panel to genes actually present in adata_de.var_names.
var_set = set(map(str, adata_de.var_names))
marker_panels_present = {}
missing_marker_rows = []

for panel, genes in marker_panels.items():
    present = [g for g in genes if g in var_set]
    missing = [g for g in genes if g not in var_set]
    if len(present) > 0:
        marker_panels_present[panel] = present
    for g in missing:
        missing_marker_rows.append({"panel": panel, "missing_gene": g})

missing_marker_panel_genes = pd.DataFrame(missing_marker_rows)
MISSING_MARKER_PANEL_TSV = METADATA_DIR / "marker_DE_HVG8000_r0_6_missing_marker_panel_genes.tsv"
write_tsv_replace(missing_marker_panel_genes, MISSING_MARKER_PANEL_TSV)

print("Number of marker panels with at least one present gene:", len(marker_panels_present))
print("Number of marker panel genes present:", sum(len(v) for v in marker_panels_present.values()))
print("Number of marker panel genes missing:", len(missing_marker_panel_genes))
if len(missing_marker_panel_genes) > 0:
    display(missing_marker_panel_genes)

if len(marker_panels_present) > 0:
    sc.pl.dotplot(
        adata_de,
        var_names=marker_panels_present,
        groupby=MAIN_CLUSTER_KEY,
        standard_scale="var",
        dendrogram=True,
        show=False,
    )
    fig = plt.gcf()
    out = MARKER_FIG_DIR / "marker_DE_known_marker_panel_dotplot.png"
    fig.savefig(out, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print("[SAVED]", out)
else:
    print("[SKIP] No known marker panel genes present for dotplot.")


In [ ]:

# =========================
# Save final h5ad with DE results in .uns
# =========================

# rank_genes_groups results are stored in adata_de.uns[RANK_KEY]
# This file is useful for re-plotting without rerunning DE.
write_h5ad_replace(adata_de, MARKER_READY_H5AD)

# Output manifest
manifest = pd.DataFrame([
    {"artifact": "input_h5ad", "path": str(FULLGENE_INPUT_H5AD), "exists": Path(FULLGENE_INPUT_H5AD).exists()},
    {"artifact": "marker_ready_h5ad", "path": str(MARKER_READY_H5AD), "exists": MARKER_READY_H5AD.exists()},
    {"artifact": "readiness_report", "path": str(DE_READINESS_TSV), "exists": DE_READINESS_TSV.exists()},
    {"artifact": "cluster_qc", "path": str(CLUSTER_QC_TSV), "exists": CLUSTER_QC_TSV.exists()},
    {"artifact": "cluster_composition", "path": str(CLUSTER_COMPOSITION_TSV), "exists": CLUSTER_COMPOSITION_TSV.exists()},
    {"artifact": "marker_panel_presence", "path": str(MARKER_PANEL_PRESENCE_TSV), "exists": MARKER_PANEL_PRESENCE_TSV.exists()},
    {"artifact": "all_de", "path": str(DE_ALL_TSV), "exists": DE_ALL_TSV.exists()},
    {"artifact": "top100_up_down", "path": str(DE_TOP100_UP_DOWN_TSV), "exists": DE_TOP100_UP_DOWN_TSV.exists()},
    {"artifact": "top20_up", "path": str(DE_TOP20_UP_TSV), "exists": DE_TOP20_UP_TSV.exists()},
    {"artifact": "filtered_de", "path": str(DE_FILTERED_TSV), "exists": DE_FILTERED_TSV.exists()},
    {"artifact": "figures_dir", "path": str(MARKER_FIG_DIR), "exists": MARKER_FIG_DIR.exists()},
])
write_tsv_replace(manifest, DE_OUTPUT_MANIFEST_TSV)
display(manifest)



## Files to send back for annotation

After running this notebook, send these TSV files:

```text
${GLIOMA_PROJECT_ROOT}/metadata/marker_DE_HVG8000_r0_6_readiness_report.tsv
${GLIOMA_PROJECT_ROOT}/metadata/marker_DE_HVG8000_r0_6_cluster_QC_summary.tsv
${GLIOMA_PROJECT_ROOT}/metadata/marker_DE_HVG8000_r0_6_cluster_composition_summary.tsv
${GLIOMA_PROJECT_ROOT}/metadata/marker_DE_HVG8000_r0_6_marker_panel_presence.tsv
${GLIOMA_PROJECT_ROOT}/metadata/marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top100_up_down.tsv
${GLIOMA_PROJECT_ROOT}/metadata/marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top20_up.tsv
${GLIOMA_PROJECT_ROOT}/metadata/marker_DE_HVG8000_r0_6_output_manifest.tsv
```

The most important file for the next annotation step is:

```text
marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top100_up_down.tsv
```

The `.h5ad` file is likely large and does **not** need to be sent.
